# A3 원본 데이터 확인

전처리 방법을 정하기 전에, 판단이 필요한 원본 데이터의 행을 셀마다 하나씩 확인합니다. 데이터는 `uv run python scripts/download_data.py`로 받은 `data/raw/`를 읽습니다. 파일·컬럼 설명은 `docs/data.md`에 있습니다.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 80)  # 긴 제목이 잘리지 않게 한 칸에 80자까지 보여 준다

# 레포 루트(pyproject.toml이 있는 폴더)를 찾는다. 노트북을 어느 폴더에서 열어도 data/raw/를 찾기 위해서다.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"

books = pd.read_csv(RAW / "books.csv")  # 책 1만 권의 정보
ratings = pd.read_csv(RAW / "ratings.csv")  # 독자가 책에 준 별점 (파일 순서 = 시간순)
to_read = pd.read_csv(RAW / "to_read.csv")  # 독자가 "읽고 싶은 책"에 담은 책

# 태그 표: book_tags에 태그 이름(tags)과 우리 책 ID·제목(books)을 붙인다.
# NOTE: book_tags는 책을 goodreads_book_id로 가리켜서, books를 거쳐야 book_id로 바뀐다.
tags = (
    pd.read_csv(RAW / "book_tags.csv")  # goodreads_book_id, tag_id, count
    .merge(pd.read_csv(RAW / "tags.csv"), on="tag_id")  # + tag_name
    .merge(books[["goodreads_book_id", "book_id", "title"]], on="goodreads_book_id")  # + book_id, title
)
TAG_COLS = ["book_id", "title", "tag_id", "tag_name", "count"]  # 태그 표에서 볼 컬럼

## 1. book_tags: count가 음수인 행

`count`는 **그 책을 그 이름의 책장(태그)에 꽂은 Goodreads 독자 수**입니다. 책이 몇 권 있는지가 아닙니다. 예를 들어 해리 포터 1권의 `owned` 6066은 "이 책을 가지고 있다(owned)"는 책장에 꽂은 독자가 6,066명이라는 뜻입니다. 사람 수라서 음수가 나올 수 없습니다.

In [ ]:
tags.loc[tags["count"] < 0, TAG_COLS]  # count < 0인 행만 고른다

음수 행이 있는 책의 태그 목록 끝부분입니다. 태그는 count가 큰 순서로 정렬되어 있어서, 음수가 count 1 다음 맨 끝에 있는지 봅니다.

In [ ]:
# 음수 행이 있는 책마다 태그 목록의 마지막 8행을 보여 준다
for book_id in tags.loc[tags["count"] < 0, "book_id"].unique():
    display(tags.loc[tags["book_id"] == book_id, TAG_COLS].tail(8))

## 2. book_tags: 같은 (책, 태그)가 여러 번 나오는 행

한 책에 같은 태그는 한 번만 있어야 합니다. 두 번 나오면 같은 책장의 독자 수가 두 번 기록된 것입니다(Goodreads 원본 XML에도 두 번 들어 있음).

In [ ]:
# keep=False: 중복된 행을 첫 번째 것까지 모두 표시한다. 같은 쌍끼리 붙여 보이게 정렬한다.
dup = tags[tags.duplicated(["book_id", "tag_id"], keep=False)].sort_values(["book_id", "tag_id"])
dup[TAG_COLS]

중복 쌍마다 두 값 중 작은 값(min), 큰 값(max), 합(sum)입니다. 결정: 큰 값 하나만 남깁니다. 합은 같은 독자를 두 번 세게 됩니다.

In [ ]:
dup.groupby(["book_id", "title", "tag_name"])["count"].agg(["min", "max", "sum"])  # (책, 태그)별로 묶어 세 값을 계산

## 3. books: 출판연도

In [ ]:
YEAR_COLS = ["book_id", "title", "authors", "original_publication_year"]
year = books["original_publication_year"]
books.loc[year.isna(), YEAR_COLS]  # 연도가 비어 있는 책

In [ ]:
books.loc[year <= 0, YEAR_COLS].sort_values("original_publication_year")  # 0 이하 = 기원전 책, 오래된 순

## 4. books: 저자

In [ ]:
# 저자 칸에 쉼표가 있으면 여러 명이다. 번역가·삽화가도 같은 칸에 들어 있다.
multi = books[books["authors"].str.contains(",")]
print(f"저자가 여러 명인 책: {len(multi)}권")
multi[["book_id", "title", "authors"]].head(10)

저자 이름 안에 공백이 2칸 이상 들어간 경우입니다(예: `Betty  Smith`). 그대로 두면 `Betty Smith`와 다른 사람으로 셉니다.

In [ ]:
names = books["authors"].str.split(",").explode().str.strip()  # 저자 칸을 쉼표로 나눠 한 줄에 한 명씩 펼친다
spaced = names[names.str.contains(r"\s{2,}")]  # 공백이 2칸 이상 연속인 이름
print(f"공백이 여러 칸인 이름: {spaced.nunique()}개, 해당 책: {spaced.index.nunique()}권")
print("공백을 한 칸으로 줄이면 저자 수:", names.nunique(), "→", names.str.replace(r"\s+", " ", regex=True).nunique())
spaced.drop_duplicates().head(10)

## 5. books: title과 original_title

In [ ]:
# title = 이 판본의 제목(시리즈 정보 포함), original_title = 처음 나왔을 때의 제목
print("original_title 결측:", books["original_title"].isna().sum(), "/ title 결측:", books["title"].isna().sum())
print("앞뒤 공백이 있는 title:", (books["title"] != books["title"].str.strip()).sum())
books[["book_id", "title", "original_title"]].head(10)

제목이 같은 책입니다. 대부분 저자가 다른 별개의 책이라, 책은 제목이 아니라 `book_id`로 구분해야 합니다.

In [ ]:
books[books["title"].duplicated(keep=False)].sort_values("title")[["book_id", "title", "authors", "original_publication_year"]]

## 6. books: language_code

In [ ]:
books["language_code"].value_counts(dropna=False)  # dropna=False: 결측(NaN)도 개수에 포함

언어가 비어 있는 책의 제목입니다. 제목이 영어 알파벳·숫자·기호만으로 되어 있는지(ASCII) 함께 봅니다.

In [ ]:
missing = books.loc[books["language_code"].isna(), ["book_id", "title", "authors"]]
print(f"결측 {len(missing)}권 중 제목이 ASCII만인 책: {missing['title'].map(str.isascii).sum()}권")
missing.sample(15, random_state=0)  # 무작위 15권 (random_state로 매번 같은 15권)

## 7. ratings: 행 순서

원본 README에 따르면 ratings는 시간순입니다. `user_id`가 오름차순으로 묶여 있지 않으면, 독자별이 아니라 전체가 하나의 시간순으로 섞여 있다는 뜻입니다.

In [ ]:
print("user_id가 오름차순으로 정렬돼 있나:", ratings["user_id"].is_monotonic_increasing)
# 독자 한 명의 평점이 파일에서 몇 군데로 흩어져 있는지 센다 (바로 윗줄과 독자가 바뀌는 곳 = 새 덩어리의 시작)
blocks = (ratings["user_id"] != ratings["user_id"].shift()).groupby(ratings["user_id"]).sum()
print("독자 한 명의 평점이 나뉜 덩어리 수 (중앙값):", blocks.median())
ratings.head(15)

## 8. ratings: 별점이 모두 같은 독자

모든 책에 같은 별점만 준 독자입니다. 평균과의 차이가 모두 0이라, 평균을 빼서 취향을 계산하는 방식(예: content-based 독자 프로필)에서 0으로 나누게 됩니다.

In [ ]:
std = ratings.groupby("user_id")["rating"].std()  # 독자별 별점의 표준편차 (0 = 모두 같은 별점)
print(f"별점이 모두 같은 독자: {(std == 0).sum()}명")
ratings[ratings["user_id"].isin(std[std == 0].index)].groupby("user_id")["rating"].agg(["first", "size"]).head(10)

## 9. to_read와 ratings에 모두 있는 (독자, 책) 쌍

`position`은 그 평점이 독자 이력에서 어디쯤인지입니다(0 = 첫 평점, 1 = 마지막 평점).

In [ ]:
order = ratings.groupby("user_id").cumcount()  # 독자별로 몇 번째 평점인지 (0부터)
size = ratings.groupby("user_id")["user_id"].transform("size")  # 그 독자의 전체 평점 수
both = to_read.merge(ratings.assign(position=(order / (size - 1)).round(2)), on=["user_id", "book_id"])  # 두 파일에 모두 있는 쌍
print(f"{len(both)}쌍, 독자 {both['user_id'].nunique()}명, 책 {both['book_id'].nunique()}권")
both.merge(books[["book_id", "title"]], on="book_id").head(20)

겹치는 쌍의 별점 분포를 전체 평점과 비교합니다. 읽고 싶다고 담은 책이 실제로 더 높은 별점을 받았는지 봅니다.

In [ ]:
# normalize=True: 개수 대신 비율
pd.DataFrame({
    "겹치는 쌍": both["rating"].value_counts(normalize=True).sort_index(),
    "전체 평점": ratings["rating"].value_counts(normalize=True).sort_index(),
}).round(3)

독자별 앞 80%를 학습으로 나눌 때 겹치는 쌍이 어느 쪽에 들어가는지입니다.

In [ ]:
in_train = both["position"] < 0.8  # 이력의 앞 80% 안에 있으면 학습 쪽
print(f"학습 쪽: {in_train.sum()}쌍 / 검증·평가 쪽: {(~in_train).sum()}쌍")